In [1]:
# -*- coding: utf-8 -*-
# Import necessary libraries
import sympy
from sympy import symbols, Matrix, Function, sin, exp, diff, latex, simplify
from IPython.display import display, Math

try:
    from gr import (
        compute_christoffel_symbols, display_christoffel_symbols,
        compute_riemann_curvature_tensor, display_riemann_curvature_tensor,
        compute_ricci_tensor, display_ricci_tensor,
        compute_ricci_scalar, display_ricci_scalar,
        compute_einstein_tensor, display_einstein_tensor
    )
    print("Successfully imported functions from 'gr' module.")
except ModuleNotFoundError:
    !git clone https://github.com/ToelUl/Einstein-tensor-calculator.git
    !cp -r Einstein-tensor-calculator/gr ./
    from gr import (
        compute_christoffel_symbols, display_christoffel_symbols,
        compute_riemann_curvature_tensor, display_riemann_curvature_tensor,
        compute_ricci_tensor, display_ricci_tensor,
        compute_ricci_scalar, display_ricci_scalar,
        compute_einstein_tensor, display_einstein_tensor
    )

Successfully imported functions from 'gr' module.


# Tutorial: Analyzing the Geometry of a 3-Sphere (S³) with `gr`

This notebook investigates the geometry of a 3-sphere ($S^3$), which is the surface of a four-dimensional hypersphere. It serves as a fundamental example of a closed, positively curved manifold and is particularly relevant in cosmology as a possible spatial geometry for the universe. We will use the `gr` module to compute its geometric tensors.

## 1. Introduction: Geometry of a 3-Sphere

A 3-sphere ($S^3$) of radius $R$ can be defined as the set of points $(x_1, x_2, x_3, x_4)$ in 4-dimensional Euclidean space ($\mathbb{R}^4$) that satisfy the equation:

$x_1^2 + x_2^2 + x_3^2 + x_4^2 = R^2$

We can parameterize points on the $S^3$ using hyperspherical coordinates $(\chi, \theta, \phi)$, where:
* $\chi$ is the primary angular coordinate (sometimes called the polar angle in 4D), ranging from $0$ to $\pi$. It represents the angle relative to the $x_1$-axis.
* $\theta$ is the standard polar angle ($0 \le \theta \le \pi$).
* $\phi$ is the standard azimuthal angle ($0 \le \phi < 2\pi$).

A common parametrization is:
$x_1 = R \cos \chi$
$x_2 = R \sin \chi \cos \theta$
$x_3 = R \sin \chi \sin \theta \cos \phi$
$x_4 = R \sin \chi \sin \theta \sin \phi$

The intrinsic geometry of the 3-sphere is described by the line element $ds^2$ obtained by restricting the 4D Euclidean metric ($ds^2 = dx_1^2 + dx_2^2 + dx_3^2 + dx_4^2$) to the surface of the sphere. This calculation yields:

$ds^2 = R^2 \left[ d\chi^2 + \sin^2\chi \left( d\theta^2 + \sin^2\theta d\phi^2 \right) \right]$

This line element represents the distance between nearby points *on* the 3-sphere. From this, we can identify the components of the 3D metric tensor $g_{\mu\nu}$ in the coordinates $(\chi, \theta, \phi)$:
* $g_{\chi\chi} = R^2$
* $g_{\theta\theta} = R^2 \sin^2\chi$
* $g_{\phi\phi} = R^2 \sin^2\chi \sin^2\theta$
* All off-diagonal components are zero.

**Key Geometric Properties:**
* **Intrinsic Curvature:** The 3-sphere is intrinsically curved with constant positive curvature. This will result in a non-zero Riemann tensor.
* **Ricci Scalar:** For an n-dimensional sphere $S^n$ of radius R, the Ricci scalar curvature is $R = \frac{n(n-1)}{R^2}$. For the 3-sphere ($n=3$), we expect $R = \frac{3(3-1)}{R^2} = \frac{6}{R^2}$.
* **Ricci Tensor:** For a manifold of constant curvature, the Ricci tensor is proportional to the metric: $R_{\mu\nu} = \frac{R}{n} g_{\mu\nu}$. For $S^3$, we expect $R_{\mu\nu} = \frac{(6/R^2)}{3} g_{\mu\nu} = \frac{2}{R^2} g_{\mu\nu}$.
* **Einstein Tensor:** Defined as $G_{\mu\nu} = R_{\mu\nu} - \frac{1}{2} g_{\mu\nu} R$. Substituting the expected values for $S^3$: $G_{\mu\nu} = \frac{2}{R^2} g_{\mu\nu} - \frac{1}{2} g_{\mu\nu} \left(\frac{6}{R^2}\right) = \left(\frac{2}{R^2} - \frac{3}{R^2}\right) g_{\mu\nu} = -\frac{1}{R^2} g_{\mu\nu}$. Unlike the 2D case where $G_{ij}$ vanished, the Einstein tensor for the 3-sphere is non-zero and proportional to the metric.

Let's compute these tensors using the `gr` module.

In [2]:
# --- Define Coordinates, Parameters, and Metric ---

# Define symbols for 3D hyperspherical coordinates (chi, theta, phi)
# Assume chi and theta in (0, pi), phi in (0, 2pi) to avoid coordinate degeneracies.
chi, theta, phi = symbols("χ θ φ", real=True, positive=True)
coords_3d = [chi, theta, phi]
print("Coordinates:")
display(coords_3d)

# Define the constant radius R (positive real number)
R = symbols("R", real=True, positive=True)
print("\nRadius Parameter:")
display(R)

# Define the metric tensor components for the 3-sphere
g_chichi = R**2
g_thetatheta = R**2 * sin(chi)**2
g_phiphi = R**2 * sin(chi)**2 * sin(theta)**2

# Construct the 3x3 metric tensor as a SymPy Matrix
metric_sphere_3d = Matrix([
    [ g_chichi,          0,                0          ],
    [     0,      g_thetatheta,             0          ],
    [     0,            0,             g_phiphi     ]
])

print("\n3-Sphere Metric Tensor g_μν:")
# Display the metric using LaTeX
display(Math(latex(metric_sphere_3d)))

# Display its components for clarity
print("\nMetric Components:")
display(Math(f"g_{{χχ}} = {latex(g_chichi)}"))
display(Math(f"g_{{θθ}} = {latex(g_thetatheta)}"))
display(Math(f"g_{{φφ}} = {latex(g_phiphi)}"))

Coordinates:


[χ, θ, φ]


Radius Parameter:


R


3-Sphere Metric Tensor g_μν:


<IPython.core.display.Math object>


Metric Components:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 2. Christoffel Symbols ($Γ^γ_{αβ}$)

The Christoffel symbols describe how the basis vectors $(\partial_\chi, \partial_\theta, \partial_\phi)$ change as we move on the 3-sphere.

$\Gamma^\gamma_{\alpha\beta} = \frac{1}{2} g^{\gamma\delta} \left( \frac{\partial g_{\beta\delta}}{\partial x^\alpha} + \frac{\partial g_{\alpha\delta}}{\partial x^\beta} - \frac{\partial g_{\alpha\beta}}{\partial x^\delta} \right)$
(Indices $\alpha, \beta, \gamma, \delta$ run over $\chi, \theta, \phi$)

In [3]:
# --- Compute and Display Christoffel Symbols ---

print("Computing Christoffel Symbols Γ^γ_{αβ}...")
# Use the 3D coordinates and 3x3 metric
christoffel_symbols_3d = compute_christoffel_symbols(coords_3d, metric_sphere_3d)

# Display the non-zero Christoffel symbols
if christoffel_symbols_3d:
    display_christoffel_symbols(christoffel_symbols_3d, coords_3d)
else:
    print("Could not compute Christoffel symbols (check 'gr' module import).")

Computing Christoffel Symbols Γ^γ_{αβ}...

--- Christoffel Symbols Γ^ρ_{μν} ---


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 3. Riemann Curvature Tensor ($R^γ_{δ α β}$)

The Riemann tensor quantifies the intrinsic curvature of the 3-sphere. We expect non-zero components due to its curvature.

$R^\gamma_{\delta\alpha\beta} = \frac{\partial \Gamma^\gamma_{\beta\delta}}{\partial x^\alpha} - \frac{\partial \Gamma^\gamma_{\alpha\delta}}{\partial x^\beta} + \Gamma^\gamma_{\alpha\epsilon} \Gamma^\epsilon_{\beta\delta} - \Gamma^\gamma_{\beta\epsilon} \Gamma^\epsilon_{\alpha\delta}$
(Indices run over $\chi, \theta, \phi$)

In [4]:
# --- Compute and Display Riemann Curvature Tensor ---

print("Computing Riemann Curvature Tensor R^γ_{δ α β}...")
# Pass the pre-computed Christoffel symbols
riemann_tensor_3d = compute_riemann_curvature_tensor(coords_3d, metric_sphere_3d, christoffel_symbols=christoffel_symbols_3d)

# Display the non-zero components
# Expect non-zero components indicating curvature
if riemann_tensor_3d:
    display_riemann_curvature_tensor(riemann_tensor_3d, coords_3d)
else:
    print("Could not compute Riemann tensor.")

Computing Riemann Curvature Tensor R^γ_{δ α β}...

--- Riemann Curvature Tensor R^ρ_{σ μ ν} ---


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 4. Ricci Curvature Tensor ($R_{αβ}$)

The Ricci tensor is the contraction $R_{\alpha\beta} = R^\gamma_{\alpha\gamma\beta}$. For the 3-sphere, due to its constant positive curvature, we expect the Ricci tensor to be proportional to the metric: $R_{\alpha\beta} = \frac{2}{R^2} g_{\alpha\beta}$.

In [5]:
# --- Compute and Display Ricci Tensor ---

print("Computing Ricci Tensor R_{αβ}...")
# Pass the pre-computed Riemann tensor
ricci_tensor_3d = compute_ricci_tensor(coords_3d, metric_sphere_3d, riemann_tensor=riemann_tensor_3d)

# Display the components
# Expect R_αβ = (2/R^2) * g_αβ
if ricci_tensor_3d:
    print("Displaying Ricci Tensor Components:")
    display_ricci_tensor(ricci_tensor_3d, coords_3d)

    # Verify the relationship R_αβ = (2/R^2) * g_αβ
    expected_ricci_3d = (2/R**2) * metric_sphere_3d
    print("\nVerifying if R_αβ = (2/R^2) * g_αβ:")
    diff_matrix_3d = sympy.simplify(Matrix(ricci_tensor_3d) - expected_ricci_3d)
    is_relation_valid_3d = (diff_matrix_3d == sympy.zeros(3, 3))

    if is_relation_valid_3d:
        print("Verification successful: R_αβ = (2/R^2) * g_αβ holds.")
        display(Math(f"R_{{αβ}} = \\frac{{2}}{{R^2}} g_{{αβ}} = {latex(expected_ricci_3d)}"))
    else:
        print("Warning: Relationship R_αβ = (2/R^2) * g_αβ did not verify!")
        print("Computed Ricci:")
        display(Matrix(ricci_tensor_3d))
        print("Expected Ricci:")
        display(expected_ricci_3d)
else:
    print("Could not compute Ricci tensor.")

Computing Ricci Tensor R_{αβ}...
Displaying Ricci Tensor Components:

--- Ricci Tensor R_{μν} ---


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>


Verifying if R_αβ = (2/R^2) * g_αβ:
Verification successful: R_αβ = (2/R^2) * g_αβ holds.


<IPython.core.display.Math object>

## 5. Ricci Scalar ($R$)

The Ricci scalar is $R = g^{\alpha\beta} R_{\alpha\beta}$. For a 3-sphere, it should be the constant positive value $R = 6/R^2$.

In [6]:
# --- Compute and Display Ricci Scalar ---

print("Computing Ricci Scalar R...")
# Pass the pre-computed Ricci tensor
ricci_scalar_3d = compute_ricci_scalar(coords_3d, metric_sphere_3d, ricci_tensor=ricci_tensor_3d)

# Display the result (expecting 6/R^2)
expected_scalar_3d = 6/R**2
if ricci_scalar_3d is not None:
    print("Displaying Ricci Scalar:")
    display_ricci_scalar(ricci_scalar_3d)
    # Explicit check
    if simplify(ricci_scalar_3d - expected_scalar_3d) == 0:
        print(f"\nVerification successful: The Ricci scalar R = 6/R^2.")
    else:
        print(f"\nWarning: The Ricci scalar did not simplify to 6/R^2! Result: {ricci_scalar_3d}")
else:
    print("Could not compute Ricci scalar.")

Computing Ricci Scalar R...
Displaying Ricci Scalar:

--- Ricci Scalar R ---


<IPython.core.display.Math object>


Verification successful: The Ricci scalar R = 6/R^2.


## 6. Einstein Tensor ($G_{αβ}$)

The Einstein tensor is $G_{\alpha\beta} = R_{\alpha\beta} - \frac{1}{2} g_{\alpha\beta} R$. For the 3-sphere, using $R_{\alpha\beta} = \frac{2}{R^2} g_{\alpha\beta}$ and $R = 6/R^2$, we expect $G_{\alpha\beta} = \frac{2}{R^2} g_{\alpha\beta} - \frac{1}{2} g_{\alpha\beta} \left(\frac{6}{R^2}\right) = -\frac{1}{R^2} g_{\alpha\beta}$. It is non-zero and proportional to the metric.

In [7]:
# --- Compute and Display Einstein Tensor ---

print("Computing Einstein Tensor G_{αβ}...")
# Pass the pre-computed Ricci tensor and scalar
einstein_tensor_3d = compute_einstein_tensor(coords_3d, metric_sphere_3d, ricci_tensor=ricci_tensor_3d, ricci_scalar=ricci_scalar_3d)

# Display the components (expecting -(1/R^2) * g_αβ)
if einstein_tensor_3d:
    print("Displaying Einstein Tensor Components:")
    display_einstein_tensor(einstein_tensor_3d, coords_3d)

    # Verify the relationship G_αβ = -(1/R^2) * g_αβ
    expected_einstein_3d = (-1/R**2) * metric_sphere_3d
    print("\nVerifying if G_αβ = -(1/R^2) * g_αβ:")
    diff_matrix_E_3d = sympy.simplify(Matrix(einstein_tensor_3d) - expected_einstein_3d)
    is_E_relation_valid_3d = (diff_matrix_E_3d == sympy.zeros(3, 3))

    if is_E_relation_valid_3d:
        print("Verification successful: G_αβ = -(1/R^2) * g_αβ holds.")
        display(Math(f"G_{{αβ}} = -\\frac{{1}}{{R^2}} g_{{αβ}} = {latex(expected_einstein_3d)}"))
    else:
        print("Warning: Relationship G_αβ = -(1/R^2) * g_αβ did not verify!")
        print("Computed Einstein:")
        display(Matrix(einstein_tensor_3d))
        print("Expected Einstein:")
        display(expected_einstein_3d)
else:
    print("Could not compute Einstein tensor.")

Computing Einstein Tensor G_{αβ}...
Displaying Einstein Tensor Components:

--- Einstein Tensor G_{μν} ---


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>


Verifying if G_αβ = -(1/R^2) * g_αβ:
Verification successful: G_αβ = -(1/R^2) * g_αβ holds.


<IPython.core.display.Math object>

## 7. Conclusion

We have computed the geometric tensors for the 3-sphere ($S^3$) using the `gr` module. The results confirm its properties as a manifold of constant positive curvature:

* The **Riemann tensor** is non-zero, reflecting the intrinsic curvature.
* The **Ricci tensor** is proportional to the metric: $R_{\alpha\beta} = \frac{2}{R^2} g_{\alpha\beta}$.
* The **Ricci scalar** is constant and positive: $R = 6/R^2$.
* The **Einstein tensor** is non-zero and proportional to the metric: $G_{\alpha\beta} = -\frac{1}{R^2} g_{\alpha\beta}$. This contrasts with the 2-sphere where the Einstein tensor vanished identically.

The geometry of the 3-sphere is fundamental in areas like topology and cosmology, where it represents the spatial section of a closed Friedmann-Lemaître-Robertson-Walker (FLRW) universe model. These calculations showcase the utility of symbolic tools in exploring such geometries.